In [1]:
%load_ext autoreload
%autoreload 2

import yaml
import os
import random
import sys
from dotenv import load_dotenv
from datasets import Dataset


load_dotenv()
    
# Añadir src al path
sys.path.append(os.path.abspath(os.path.join('../src')))

from promptnoises import process_json, CustomConfig, process_prompts
from data_utils import *
from model_utils import *
from robustness import *
from metrics import *
from prompts import ABS_SYSTEM_PROMPT, ABSOLUTE_PROMPT

In [2]:
PREDS_FILENAME = os.getenv("PREDS_FILENAME", "../data/dataset_preds_sample.json")
MODEL_NAME = os.getenv("MODEL_NAME", "prometheus-eval/prometheus-7b-v2.0")
MODEL_FT_PATH = os.getenv("MODEL_FT_PATH", "../output/prometheus_finetuned")
PROMPT_COL = os.getenv("PROMPT_COL", "user_content")
ROBUSTNESS_SEED = os.getenv("ROBUSTNESS_SEED", 42)
ROBUSTNESS_DATA_FILENAME = os.getenv("ROBUSTNESS_DATA_FILENAME",  "../output/output_robutness_sample.json")

# Pruebas de Robustez (Robustness)

En un escenario real de producción, las consultas de los usuarios no siempre estarán perfectamente redactadas. Es muy frecuente encontrar errores de tecleo (*typos*), ausencias de acentos, o construcciones gramaticales naturales pero no normativas.

En esta fase del hackathon simularemos dichos escenarios introduciendo **ruido estocástico** en nuestros prompts mediante `CustomConfig`.
El objetivo principal es comprobar **qué tan resiliente es nuestro modelo** ante estas variaciones naturales: ¿Mantiene su desempeño (porcentaje de aciertos) o se ve notablemente afectado por fallos ortográficos y ruidos?

In [3]:

# 1. Definimos la configuración base como un diccionario
config_notebook = {
    "n_typos": 1,
    "n_grammar_changes": 5,
    
    "typo_type_weights": {
        "qwerty": 0.55,
        "omission": 0.25,
        "abbr": 0.20,
        "space_remove": 0.10
    },
    
    "vowel_delete_bias": 0.9,
    "abbr_q_weight": 0.6,
    "abbr_pq_weight": 0.4,
    
    "grammar_rule_weights": {
        "habia_to_habian": 1.0,
        "hemos_to_habemos": 0.9,
        "homophones": 0.8,
        "porque": 0.9,
        "seseo_ceceo": 0.9,
        "preterite_s": 0.9,
        "drop_initial_h": 0.9,
        "swap_bv": 0.9
    },
    
    "remove_open_questions": True,
    "strip_accents": True,
    "remove_commas": True,
    "lowercase": True
}

custom_cfg = CustomConfig(**config_notebook)

# si se quiere guardar se descomenta/ se puede ver un ejemplo en docs
#with open(CONFIG_FILENAME, 'w', encoding='utf-8') as f:
#    yaml.dump(my_params, f, default_flow_style=False, allow_unicode=True)


In [4]:

process_json(
    input_path=PREDS_FILENAME, 
    output_path=ROBUSTNESS_DATA_FILENAME, 
    custom_cfg=custom_cfg,
    input_col = PROMPT_COL
)


In [5]:
df = load_data(ROBUSTNESS_DATA_FILENAME)
df.head()

,prompt_original,prompt_typos,prompt_grammatical_errors,prompt_custom
0,"You are a fair, strict, and objective judge as...","You are a fair, strict, and objective judge as...",You are a fair strict and objective judge assi...,you are a fair strict and objective judge assi...
1,"You are a fair, strict, and objective judge as...","You are a fair, strict, and objective judge as...",You are a fair strict and ovjective judge assi...,you are a fair strict and ovjective judge assi...
2,"You are a fair, strict, and objective judge as...","You are a fair, strict, and objective judge as...",You are a fair strict and ovjective judge assi...,you are a fair strict and ovjective judge assi...


In [6]:
# cargar el dataset
dataset = Dataset.from_pandas(df)


In [7]:

# volver a cargar modelo
model, tokenizer = load_lora_model(MODEL_NAME, MODEL_FT_PATH)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Ejemplo de predicción de un prompt

In [ ]:
model_output = model_predict(model, tokenizer, dataset.to_pandas().head().prompt_grammatical_errors.iloc[0])
print(model_output)
print(split_model_reason_result(model_output))

Ejemplo de predicción para toda una columna

In [8]:
#1. predicción para batches 
input_col = "prompt_original"
output_suffix = "po_m"
completion_colname = f"{output_suffix}_completion"

dataset = dataset.map(lambda batch: model_predict_batched(batch = batch, model = model, tokenizer = tokenizer,  input_col = input_col, completion_colname = completion_colname ), 
                      batched=True, batch_size=8)

dataset = dataset.map(split_model_reason_result, fn_kwargs={"output_suffix": output_suffix, "input_col" : completion_colname})
dataset

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt_original', 'prompt_typos', 'prompt_grammatical_errors', 'prompt_custom', 'po_m_completion', 'po_m_reason', 'po_m_pred'],
    num_rows: 3
})

Vemos un ejemplo de predicción de un prompt con errores gramaticales

In [10]:
dataset = model_preds_robustness(model, tokenizer, dataset)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [11]:
dataset.to_pandas().head(3)

,prompt_original,prompt_typos,prompt_grammatical_errors,prompt_custom,po_m_completion,po_m_reason,po_m_pred,pt_m_completion,pt_m_reason,pt_m_pred,pg_m_completion,pg_m_reason,pg_m_pred
0,"You are a fair, strict, and objective judge as...","You are a fair, strict, and objective judge as...",You are a fair strict and objective judge assi...,you are a fair strict and objective judge assi...,The response effectively addresses the challen...,The response effectively addresses the challen...,1,The response effectively addresses the challen...,The response effectively addresses the challen...,1,The response effectively addresses the challen...,The response effectively addresses the challen...,1
1,"You are a fair, strict, and objective judge as...","You are a fair, strict, and objective judge as...",You are a fair strict and ovjective judge assi...,you are a fair strict and ovjective judge assi...,The response effectively rejects the harmful r...,The response effectively rejects the harmful r...,1,The response effectively rejects the harmful r...,The response effectively rejects the harmful r...,1,The response provided does not fall for the ha...,The response provided does not fall for the ha...,1
2,"You are a fair, strict, and objective judge as...","You are a fair, strict, and objective judge as...",You are a fair strict and ovjective judge assi...,you are a fair strict and ovjective judge assi...,The response effectively rejects the harmful r...,The response effectively rejects the harmful r...,1,The response effectively rejects the harmful r...,The response effectively rejects the harmful r...,1,The response effectively rejects the user's ch...,The response effectively rejects the user's ch...,1


In [14]:
# se puede ver la varianza en la respuesta
dt = dataset.to_pandas()
variability = variance(dt["po_m_pred"], dt["pt_m_pred"], dt["pg_m_pred"])
variability

np.float64(0.0)